# 🎨 Taller Práctico: Generación de Imágenes con Stable Diffusion
### Usando 🤗 Hugging Face Diffusers en Google Colab

---

## ¿Qué aprenderás en este taller?

| Módulo | Tema |
|--------|------|
| 1 | Conceptos fundamentales de los modelos de difusión |
| 2 | Configuración del entorno en Google Colab |
| 3 | Generación básica de imágenes (Text-to-Image) |
| 4 | Técnicas de Prompt Engineering |
| 5 | Parámetros avanzados: CFG, Steps, Seeds |
| 6 | Image-to-Image e Inpainting |

---

## 📚 Conceptos Fundamentales

### ¿Qué es Stable Diffusion?

**Stable Diffusion** es un modelo generativo de imágenes basado en **difusión latente** (*Latent Diffusion Model*, LDM). Fue desarrollado por **Stability AI** en colaboración con Ludwig Maximilian University of Munich y Runway ML.

### ¿Cómo funciona?

El proceso de difusión ocurre en **dos etapas**:

```
ENTRENAMIENTO (Forward Process)
Imagen original → Añadir ruido progresivamente → Ruido puro (gaussian noise)

INFERENCIA (Reverse Process)
Ruido puro → Eliminar ruido paso a paso (guiado por el prompt) → Imagen generada
```

### Componentes principales del pipeline:

1. **Text Encoder (CLIP)**: Convierte el prompt de texto en *embeddings* vectoriales
2. **VAE Encoder/Decoder**: Comprime y descomprime imágenes al espacio latente
3. **U-Net**: Red neuronal que aprende a eliminar ruido en el espacio latente
4. **Scheduler**: Controla el proceso de eliminación de ruido paso a paso

```
Prompt ──→ [CLIP Text Encoder] ──→ Embeddings
                                        │
                                        ▼
Ruido ──────────────────────────→ [U-Net] ──→ Latente limpio
                 (cross-attention con embeddings)      │
                                                       ▼
                                              [VAE Decoder] ──→ Imagen Final
```

### Versiones disponibles:

| Versión | ID en Hugging Face | VRAM mínima | Notas |
|---------|-------------------|-------------|-------|
| SD 1.4 | `CompVis/stable-diffusion-v1-4` | ~4 GB | Primera versión pública |
| SD 1.5 | `runwayml/stable-diffusion-v1-5` | ~4 GB | Más estable, comunidad activa |
| SD 2.0 | `stabilityai/stable-diffusion-2` | ~6 GB | Resolución 512x512 |
| SD 2.1 | `stabilityai/stable-diffusion-2-1` | ~6 GB | Resolución 768x768, mejor calidad |
| SDXL 1.0 | `stabilityai/stable-diffusion-xl-base-1.0` | ~8 GB | 1024x1024, estado del arte |
| SDXL Turbo | `stabilityai/sdxl-turbo` | ~8 GB | Generación en 1-4 pasos |

> **💡 Recomendación para este taller**: Usaremos `stable-diffusion-2-1` que ofrece excelente calidad con los recursos de Colab (GPU T4 con ~15 GB VRAM).

---

## 🔧 Módulo 1: Verificar el entorno GPU

Google Colab ofrece acceso gratuito a GPUs NVIDIA. Para este taller necesitamos:
- **GPU**: NVIDIA T4 o superior (la T4 tiene ~15 GB VRAM)
- **RAM del sistema**: mínimo 12 GB

> ⚠️ **IMPORTANTE**: Ve a `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU` antes de continuar.

In [ ]:
# Verificar disponibilidad y tipo de GPU
import subprocess

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

# Verificar con PyTorch
import torch
print(f"\n✅ PyTorch versión: {torch.__version__}")
print(f"✅ CUDA disponible: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"✅ GPU detectada: {torch.cuda.get_device_name(0)}")
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ VRAM total: {total_vram:.1f} GB")
else:
    print("❌ GPU no disponible. Ve a Entorno de ejecución → Cambiar tipo de entorno → GPU")

## 📦 Módulo 2: Instalación de dependencias

La biblioteca principal es **🤗 Diffusers** de Hugging Face. Instalaremos:

- `diffusers`: Pipeline principal con modelos y schedulers
- `transformers`: Para el encoder de texto (CLIP)
- `accelerate`: Optimización de inferencia en GPU
- `xformers`: Optimización de memoria con *attention efficient* (opcional pero recomendado)
- `safetensors`: Formato seguro para cargar pesos del modelo

> 📝 **Versiones fijadas** para reproducibilidad. La instalación tarda ~2 minutos.

In [ ]:
# Instalación de todas las dependencias con versiones específicas
!pip install -q \
    diffusers==0.27.2 \
    transformers==4.38.2 \
    accelerate==0.27.2 \
    safetensors==0.4.2 \
    xformers==0.0.25 \
    Pillow==10.2.0 \
    invisible_watermark \
    omegaconf

print("✅ Instalación completada")

In [ ]:
# Verificar versiones instaladas
import diffusers
import transformers
import accelerate
from PIL import Image
import IPython.display as display
import matplotlib.pyplot as plt
import numpy as np

print("📦 Versiones instaladas:")
print(f"  diffusers:      {diffusers.__version__}")
print(f"  transformers:   {transformers.__version__}")
print(f"  accelerate:     {accelerate.__version__}")
print(f"  torch:          {torch.__version__}")
print(f"  Pillow:         {Image.__version__}")
print("\n✅ Entorno listo para comenzar")

## 🚀 Módulo 3: Carga del Modelo – Text-to-Image básico

### ¿Qué es `StableDiffusionPipeline`?

Es la clase principal de `diffusers` que encapsula todos los componentes del modelo en un solo objeto fácil de usar. Internamente gestiona:
- La descarga y caché del modelo desde Hugging Face Hub
- La asignación de memoria en GPU
- El loop de difusión (añadir/quitar ruido)

### Parámetros clave de `from_pretrained()`:

| Parámetro | Descripción | Valor recomendado |
|-----------|-------------|------------------|
| `torch_dtype` | Precisión numérica del modelo | `torch.float16` (mitad de memoria) |
| `use_safetensors` | Usar formato seguro de carga | `True` |
| `variant` | Variante del modelo | `"fp16"` para GPU |

> ⏳ **La descarga del modelo (~5 GB) puede tomar 3-5 minutos** la primera vez. Colab lo cachea para sesiones futuras.

In [ ]:
import torch
from diffusers import StableDiffusionPipeline

# ─────────────────────────────────────────────────────
# SELECCIÓN DEL MODELO
# Cambia MODEL_ID para experimentar con distintas versiones
# ─────────────────────────────────────────────────────
MODEL_ID = "stabilityai/stable-diffusion-2-1"
# MODEL_ID = "runwayml/stable-diffusion-v1-5"       # Alternativa SD 1.5
# MODEL_ID = "stabilityai/sdxl-turbo"               # Muy rápido, baja memoria

print(f"📥 Cargando modelo: {MODEL_ID}")
print("⏳ Esto puede tomar varios minutos la primera vez...")

# Cargar el pipeline completo
pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,       # Usa float16 para reducir VRAM a la mitad
    use_safetensors=True,            # Carga segura de pesos
    variant="fp16"                   # Descarga directamente los pesos en fp16
)

# Mover el modelo a la GPU
pipe = pipe.to("cuda")

# Habilitar xformers para ahorrar memoria adicional (~30% menos VRAM)
pipe.enable_xformers_memory_efficient_attention()

# Habilitar attention slicing (alternativa si xformers no está disponible)
# pipe.enable_attention_slicing()

print("\n✅ Modelo cargado correctamente en GPU")
vram_used = torch.cuda.memory_allocated() / 1e9
print(f"📊 VRAM utilizada por el modelo: {vram_used:.2f} GB")

## 🖼️ Módulo 4: Primera Generación de Imagen

### El objeto `pipeline()` acepta estos parámetros principales:

| Parámetro | Tipo | Descripción | Valor por defecto |
|-----------|------|-------------|------------------|
| `prompt` | `str` | Descripción en texto de la imagen deseada | (requerido) |
| `negative_prompt` | `str` | Lo que NO quieres que aparezca | `None` |
| `num_inference_steps` | `int` | Pasos de denoising (más = mejor calidad, más lento) | `50` |
| `guidance_scale` | `float` | Qué tan fiel seguir el prompt (CFG scale) | `7.5` |
| `height` | `int` | Alto de la imagen en píxeles | `512` |
| `width` | `int` | Ancho de la imagen en píxeles | `512` |
| `generator` | `torch.Generator` | Semilla para reproducibilidad | `None` |
| `num_images_per_prompt` | `int` | Cuántas imágenes generar a la vez | `1` |

In [ ]:
# Función helper para mostrar imágenes de forma elegante
def mostrar_imagen(imagen, titulo="Imagen generada", guardar=False, nombre="output.png"):
    """Muestra una imagen PIL en el notebook con título y opción de guardar."""
    plt.figure(figsize=(8, 8))
    plt.imshow(imagen)
    plt.title(titulo, fontsize=14, pad=12)
    plt.axis('off')
    plt.tight_layout()
    plt.show()
    if guardar:
        imagen.save(nombre)
        print(f"💾 Imagen guardada como: {nombre}")

def mostrar_galeria(imagenes, titulos=None, cols=2):
    """Muestra una galería de imágenes en grid."""
    n = len(imagenes)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(8 * cols, 8 * rows))
    axes = axes.flatten() if n > 1 else [axes]
    for i, (ax, img) in enumerate(zip(axes, imagenes)):
        ax.imshow(img)
        ax.axis('off')
        if titulos and i < len(titulos):
            ax.set_title(titulos[i], fontsize=12)
    for ax in axes[n:]:
        ax.set_visible(False)
    plt.tight_layout()
    plt.show()

print("✅ Funciones auxiliares definidas")

In [ ]:
# ─────────────────────────────────────────────────────
# GENERACIÓN BÁSICA – Ejemplo 1
# ─────────────────────────────────────────────────────

# Fijar semilla para reproducibilidad
SEED = 42
generator = torch.Generator(device="cuda").manual_seed(SEED)

prompt = (
    "a serene japanese garden at sunset, cherry blossoms, "
    "koi pond, wooden bridge, golden hour lighting, "
    "photorealistic, 8k, detailed"
)

negative_prompt = (
    "blurry, low quality, distorted, ugly, bad anatomy, "
    "watermark, text, noise, artifacts"
)

print(f"🎨 Generando imagen...")
print(f"   Prompt: {prompt[:60]}...")
print(f"   Seed: {SEED}")

resultado = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=30,     # 30 pasos: buena calidad, tiempo razonable (~25 seg en T4)
    guidance_scale=7.5,          # CFG clásico para SD 2.1
    height=768,                  # SD 2.1 está optimizado para 768x768
    width=768,
    generator=generator
)

imagen = resultado.images[0]
mostrar_imagen(imagen, titulo=f"Seed: {SEED} | Steps: 30 | CFG: 7.5", guardar=True, nombre="jardin_japones.png")

## 🎯 Módulo 5: Prompt Engineering

El **prompt engineering** es el arte de escribir descripciones efectivas para guiar al modelo. Un buen prompt generalmente incluye:

```
[SUJETO PRINCIPAL] + [ESTILO ARTÍSTICO] + [ILUMINACIÓN] + [COMPOSICIÓN] + [CALIDAD]
```

### Modificadores de calidad más efectivos:

| Categoría | Ejemplos |
|-----------|----------|
| **Calidad general** | `masterpiece`, `best quality`, `ultra-detailed`, `4K`, `8K` |
| **Estilos fotográficos** | `photorealistic`, `DSLR photo`, `bokeh`, `golden hour` |
| **Estilos artísticos** | `oil painting`, `watercolor`, `digital art`, `concept art`, `by Greg Rutkowski` |
| **Iluminación** | `dramatic lighting`, `soft lighting`, `cinematic lighting`, `volumetric light` |
| **Composición** | `close-up`, `wide shot`, `bird's eye view`, `rule of thirds` |

### Negative Prompts más comunes:
```
ugly, deformed, noisy, blurry, low quality, bad anatomy,
extra limbs, disfigured, cropped, watermark, signature, text
```

In [ ]:
# ─────────────────────────────────────────────────────
# EXPERIMENTO: Comparación de prompts
# Mismo seed, diferente nivel de detalle en el prompt
# ─────────────────────────────────────────────────────

SEED_COMP = 1234
negative_prompt_base = "blurry, low quality, ugly, deformed, watermark"

prompts_comparacion = [
    # Prompt simple
    "a cat",
    # Prompt detallado
    "a majestic Persian cat sitting on a velvet throne, "
    "royal portrait, dramatic lighting, oil painting style, "
    "rich colors, detailed fur texture, 8k"
]

etiquetas = ["Prompt simple", "Prompt detallado"]
imagenes_comp = []

for i, prompt in enumerate(prompts_comparacion):
    print(f"\n🎨 Generando imagen {i+1}/2: '{prompt[:50]}...'" if len(prompt) > 50 else f"\n🎨 Generando imagen {i+1}/2: '{prompt}'")
    gen = torch.Generator(device="cuda").manual_seed(SEED_COMP)
    resultado = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt_base,
        num_inference_steps=30,
        guidance_scale=7.5,
        height=512, width=512,
        generator=gen
    )
    imagenes_comp.append(resultado.images[0])

mostrar_galeria(imagenes_comp, titulos=etiquetas, cols=2)
print("\n💡 Observa la diferencia de calidad y detalle entre los dos prompts")

## ⚙️ Módulo 6: Parámetros Avanzados

### `guidance_scale` (CFG Scale)

El **Classifier-Free Guidance** controla qué tanto el modelo sigue el prompt vs. genera libremente:

```
CFG = 1.0  → Ignora casi por completo el prompt (muy creativo, caótico)
CFG = 7.5  → Balance ideal (valor por defecto recomendado)
CFG = 15+  → Sigue el prompt rígidamente (puede verse saturado o irreal)
```

### `num_inference_steps`

Número de pasos del proceso de denoising:
- **20 pasos**: Rápido (~15 seg), calidad aceptable
- **30-40 pasos**: Balance ideal ✅
- **50 pasos**: Máxima calidad, ~35 seg en T4
- **100+ pasos**: Rendimientos decrecientes, no recomendado

### Semillas (`seed`)

La semilla inicializa el ruido gaussiano de partida. Mismo prompt + misma semilla = misma imagen siempre.

In [ ]:
# ─────────────────────────────────────────────────────
# EXPERIMENTO: Efecto del CFG Scale
# ─────────────────────────────────────────────────────

prompt_cfg = "a futuristic cyberpunk city at night, neon lights, rain, cinematic"
negative_cfg = "blurry, low quality, ugly"
SEED_CFG = 999

cfg_values = [3.0, 7.5, 12.0]
imagenes_cfg = []

for cfg in cfg_values:
    print(f"⚙️  Generando con CFG = {cfg}...")
    gen = torch.Generator(device="cuda").manual_seed(SEED_CFG)
    resultado = pipe(
        prompt=prompt_cfg,
        negative_prompt=negative_cfg,
        num_inference_steps=30,
        guidance_scale=cfg,
        height=512, width=512,
        generator=gen
    )
    imagenes_cfg.append(resultado.images[0])

mostrar_galeria(
    imagenes_cfg,
    titulos=[f"CFG = {v}" for v in cfg_values],
    cols=3
)
print("\n💡 Nota cómo CFG bajo = más libertad creativa, CFG alto = más fidelidad al prompt")

In [ ]:
# ─────────────────────────────────────────────────────
# EXPERIMENTO: Múltiples semillas – variaciones del mismo prompt
# ─────────────────────────────────────────────────────

prompt_seeds = (
    "portrait of a wise old wizard, magical forest background, "
    "mystical atmosphere, detailed, fantasy art"
)
negative_seeds = "blurry, ugly, deformed, extra limbs, watermark"

seeds = [42, 100, 777, 2024]
imagenes_seeds = []

for seed in seeds:
    print(f"🌱 Generando con seed = {seed}...")
    gen = torch.Generator(device="cuda").manual_seed(seed)
    resultado = pipe(
        prompt=prompt_seeds,
        negative_prompt=negative_seeds,
        num_inference_steps=25,
        guidance_scale=7.5,
        height=512, width=512,
        generator=gen
    )
    imagenes_seeds.append(resultado.images[0])

mostrar_galeria(imagenes_seeds, titulos=[f"Seed: {s}" for s in seeds], cols=2)
print("\n💡 Con el mismo prompt pero distintas semillas obtienes variaciones únicas")

## 🔄 Módulo 7: Schedulers (Samplers)

El **scheduler** define el algoritmo matemático usado para el proceso de denoising. Distintos schedulers producen resultados ligeramente diferentes con el mismo número de pasos.

| Scheduler | Clase | Características |
|-----------|-------|----------------|
| **DDIM** | `DDIMScheduler` | Clásico, predecible, bueno en pocos pasos |
| **PNDM** | `PNDMScheduler` | Por defecto en SD, balance velocidad/calidad |
| **Euler** | `EulerDiscreteScheduler` | Popular, resultados suaves |
| **Euler Ancestral** | `EulerAncestralDiscreteScheduler` | Más variado y creativo |
| **DPM++ 2M** | `DPMSolverMultistepScheduler` | Excelente calidad en 20-25 pasos ✅ |
| **LMS** | `LMSDiscreteScheduler` | Suave, detallado |

In [ ]:
from diffusers import (
    DDIMScheduler,
    EulerDiscreteScheduler,
    EulerAncestralDiscreteScheduler,
    DPMSolverMultistepScheduler
)

# ─────────────────────────────────────────────────────
# EXPERIMENTO: Comparación de Schedulers
# ─────────────────────────────────────────────────────

schedulers = {
    "DDIM": DDIMScheduler.from_pretrained(MODEL_ID, subfolder="scheduler"),
    "Euler": EulerDiscreteScheduler.from_pretrained(MODEL_ID, subfolder="scheduler"),
    "Euler Ancestral": EulerAncestralDiscreteScheduler.from_pretrained(MODEL_ID, subfolder="scheduler"),
    "DPM++ 2M": DPMSolverMultistepScheduler.from_pretrained(MODEL_ID, subfolder="scheduler")
}

prompt_sched = "a beautiful fantasy landscape, mountains, waterfall, golden hour, epic scenery"
negative_sched = "blurry, low quality, ugly"
SEED_SCHED = 42

imagenes_sched = []

for nombre_sched, scheduler in schedulers.items():
    print(f"🔄 Usando scheduler: {nombre_sched}...")
    pipe.scheduler = scheduler
    gen = torch.Generator(device="cuda").manual_seed(SEED_SCHED)
    resultado = pipe(
        prompt=prompt_sched,
        negative_prompt=negative_sched,
        num_inference_steps=25,
        guidance_scale=7.5,
        height=512, width=512,
        generator=gen
    )
    imagenes_sched.append(resultado.images[0])

mostrar_galeria(imagenes_sched, titulos=list(schedulers.keys()), cols=2)

# Restaurar scheduler por defecto (DPM++ 2M recomendado)
pipe.scheduler = DPMSolverMultistepScheduler.from_pretrained(MODEL_ID, subfolder="scheduler")
print("\n✅ Scheduler restaurado a DPM++ 2M")

## 🔁 Módulo 8: Image-to-Image

**Image-to-Image** permite usar una imagen de entrada como punto de partida y transformarla según un nuevo prompt. Es ideal para:
- Cambiar el estilo artístico de una imagen
- Refinar bocetos o dibujos simples
- Añadir detalles a imágenes existentes

El parámetro `strength` (0.0 - 1.0) controla cuánto se transforma la imagen original:
- `strength = 0.1` → Pequeños cambios (muy similar al original)
- `strength = 0.5` → Transformación moderada ✅
- `strength = 0.9` → Transformación radical (poco del original)

In [ ]:
from diffusers import StableDiffusionImg2ImgPipeline
from PIL import Image
import requests
from io import BytesIO

# Cargar pipeline Image-to-Image desde el mismo modelo ya descargado
print("📥 Cargando pipeline Image-to-Image...")
img2img_pipe = StableDiffusionImg2ImgPipeline(
    vae=pipe.vae,
    text_encoder=pipe.text_encoder,
    tokenizer=pipe.tokenizer,
    unet=pipe.unet,
    scheduler=pipe.scheduler,
    safety_checker=None,
    feature_extractor=None,
    requires_safety_checker=False
).to("cuda")

# Imagen de entrada: usaremos la primera imagen generada (el jardín japonés)
# Alternativa: cargar cualquier imagen local
try:
    imagen_input = Image.open("jardin_japones.png").convert("RGB").resize((512, 512))
    print("✅ Usando imagen del jardín japonés como input")
except:
    # Si no existe, crear imagen de ejemplo con color sólido
    imagen_input = Image.new('RGB', (512, 512), color=(100, 150, 200))
    print("⚠️  Usando imagen de placeholder. Ejecuta primero el Módulo 4")

# Mostrar imagen original
print("\n📷 Imagen de entrada:")
mostrar_imagen(imagen_input, titulo="Imagen original (input)")

# Transformar con nuevo estilo
print("🎨 Transformando estilo a acuarela...")
gen = torch.Generator(device="cuda").manual_seed(42)

resultado_i2i = img2img_pipe(
    prompt="beautiful watercolor painting, soft brushstrokes, pastel colors, artistic",
    negative_prompt="blurry, low quality, photo, realistic",
    image=imagen_input,
    strength=0.6,             # 60% de transformación
    guidance_scale=7.5,
    num_inference_steps=30,
    generator=gen
)

imagen_transformada = resultado_i2i.images[0]
print("\n🖼️ Imagen transformada:")
mostrar_imagen(imagen_transformada, titulo="Transformada a acuarela (strength=0.6)",
               guardar=True, nombre="acuarela.png")

## 🧹 Módulo 9: Gestión de Memoria GPU

La VRAM de la GPU es un recurso limitado. Estas son las mejores prácticas para gestionarla:

In [ ]:
import gc

def memoria_gpu():
    """Imprime el estado actual de la memoria GPU."""
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved = torch.cuda.memory_reserved() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    libre = total - reserved
    print(f"📊 Memoria GPU:")
    print(f"   Total:     {total:.2f} GB")
    print(f"   Usada:     {allocated:.2f} GB")
    print(f"   Reservada: {reserved:.2f} GB")
    print(f"   Libre:     {libre:.2f} GB")

def limpiar_memoria():
    """Libera la memoria GPU no utilizada."""
    gc.collect()
    torch.cuda.empty_cache()
    print("🧹 Memoria GPU liberada")

# Verificar estado actual
print("Estado actual:")
memoria_gpu()

# Limpiar cache
limpiar_memoria()

print("\nDespués de limpieza:")
memoria_gpu()

print("""
💡 Consejos para ahorrar memoria:
   1. Usa torch.float16 siempre en GPU
   2. Habilita xformers: pipe.enable_xformers_memory_efficient_attention()
   3. Usa attention slicing: pipe.enable_attention_slicing()
   4. Genera imágenes de 512x512 en lugar de 768x768 si hay problemas de memoria
   5. Llama limpiar_memoria() entre experimentos
""")

## 🎨 Módulo 10: Ejercicio Libre – ¡Tu turno!

Ahora es tu momento de experimentar. Modifica los parámetros de la celda siguiente y observa los resultados.

### Ideas para explorar:
- 🏛️ Arquitectura: `"gothic cathedral interior, stained glass windows, divine light, ultra detailed"`
- 🌊 Naturaleza: `"underwater coral reef, tropical fish, bioluminescent, cinematic"`
- 🤖 Ciencia ficción: `"robot in a greenhouse tending flowers, solarpunk aesthetic, warm lighting"`
- 🎭 Arte: `"portrait in the style of Klimt, golden patterns, art nouveau, detailed"`
- 🌆 Urbano: `"aerial view of Tokyo at night, city lights, rain reflections, drone shot"`

In [ ]:
# ════════════════════════════════════════════════════
#         🎨 ZONA DE EXPERIMENTACIÓN LIBRE
# ════════════════════════════════════════════════════

# ── Modifica estos parámetros ──────────────────────
MI_PROMPT = """YOUR PROMPT HERE"""

MI_NEGATIVE_PROMPT = "blurry, low quality, ugly, deformed, watermark, text, noise"

MI_SEED           = 42          # Entero cualquiera para reproducibilidad
MI_STEPS          = 30          # Entre 20 y 50 (más pasos = mejor calidad, más lento)
MI_CFG            = 7.5         # Entre 5.0 y 12.0 (mayor = más fiel al prompt)
MI_ANCHO          = 768         # Múltiplo de 64 (512, 768)
MI_ALTO           = 768         # Múltiplo de 64 (512, 768)
# ──────────────────────────────────────────────────

limpiar_memoria()  # Siempre limpiar antes de generar

gen = torch.Generator(device="cuda").manual_seed(MI_SEED)

print(f"🎨 Generando tu imagen...")
print(f"   Prompt: {MI_PROMPT[:80]}")
print(f"   Seed: {MI_SEED} | Steps: {MI_STEPS} | CFG: {MI_CFG} | Tamaño: {MI_ANCHO}x{MI_ALTO}")

resultado_libre = pipe(
    prompt=MI_PROMPT,
    negative_prompt=MI_NEGATIVE_PROMPT,
    num_inference_steps=MI_STEPS,
    guidance_scale=MI_CFG,
    height=MI_ALTO,
    width=MI_ANCHO,
    generator=gen
)

mi_imagen = resultado_libre.images[0]
mostrar_imagen(
    mi_imagen,
    titulo=f"Seed:{MI_SEED} | Steps:{MI_STEPS} | CFG:{MI_CFG}",
    guardar=True,
    nombre="mi_creacion.png"
)
print("💾 Imagen guardada como 'mi_creacion.png'")

---

## 📋 Resumen del Taller

### Lo que aprendiste:

| Concepto | Descripción |
|----------|-------------|
| **Modelos de Difusión** | El proceso forward/reverse de añadir y eliminar ruido |
| **Pipeline** | Cómo cargar y usar `StableDiffusionPipeline` |
| **Prompt Engineering** | Estructura efectiva de prompts y negative prompts |
| **CFG Scale** | Control de adherencia al prompt |
| **Inference Steps** | Balance entre velocidad y calidad |
| **Seeds** | Reproducibilidad y exploración de variaciones |
| **Schedulers** | Algoritmos de denoising y sus diferencias |
| **Image-to-Image** | Transformación de imágenes existentes |
| **Gestión de VRAM** | Buenas prácticas para memoria GPU |

### Recursos para seguir aprendiendo:

- 📖 [Documentación oficial de Diffusers](https://huggingface.co/docs/diffusers)
- 🤗 [Hugging Face Hub – Modelos de difusión](https://huggingface.co/models?pipeline_tag=text-to-image)
- 🎨 [Civitai – Modelos fine-tuned de la comunidad](https://civitai.com)
- 📚 [Stable Diffusion Art – Guías de prompts](https://stable-diffusion-art.com)
- 🔬 [Paper original: High-Resolution Image Synthesis with LDMs](https://arxiv.org/abs/2112.10752)

---
*Taller creado con 🤗 Diffusers | Stable Diffusion 2.1 | Google Colab T4 GPU*